# LSL Stream - Outlet and Inlet Demo

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

This notebook demonstrates **Lab Streaming Layer (LSL)** stream creation and reception. We create a simulated EEG stream outlet, push 4-channel EEG data in chunks, then create an inlet to receive the data back. If pylsl is unavailable (common in Colab), we fall back to a simulation that copies the data directly.

## What this notebook does

1. Loads all 4 channels from the local EEG dataset
2. Creates an LSL outlet (name="SimulatedEEG", type="EEG", 4 channels, 200 Hz)
3. Pushes data in chunks of 50 samples to the outlet
4. Creates an inlet and receives the data back
5. Falls back to simulation if pylsl is unavailable

## What you should expect to see

- The top plot shows the **original 4-channel signal** (first 5000 samples, offset for visibility)
- The bottom plot shows the **received 4-channel signal** via LSL (first 5000 samples, offset for visibility)
- If real LSL is used, the signals should be identical
- If simulation mode is used, the data is copied directly (still identical)

## Key parameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| FS | 200 Hz | Sampling rate |
| CHANNEL_COUNT | 4 | Number of EEG channels |
| CHUNK_SIZE | 50 | Samples per push chunk |
| N_PLOT | 5000 | Number of samples to plot |
| Stream name | SimulatedEEG | LSL stream identifier |

## 1. Install dependencies

In [ ]:
!pip install scipy numpy plotly wfdb pylsl

## 2. Clone repo and download data

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, all 4 channels (P4, Cz, F8, T7).

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal shape: {eeg_data.shape}')
print(f'Duration: {eeg_data.shape[0]/fs:.1f} seconds')

## 4. Apply the analysis

We create an LSL outlet, push the 4-channel data in chunks of 50 samples, then create an inlet to receive the data back. If pylsl is unavailable, we fall back to a simulation.

In [ ]:
CHUNK_SIZE = 50
CHANNEL_COUNT = 4
N_PLOT = 5000

used_real_lsl = False
received_data = None

try:
    from pylsl import StreamInfo, StreamOutlet, StreamInlet, resolve_byprop
    import time

    info = StreamInfo(
        name='SimulatedEEG', type='EEG',
        channel_count=CHANNEL_COUNT, nominal_srate=fs,
        channel_format='float32',
    )
    outlet = StreamOutlet(info)

    n_samples = eeg_data.shape[0]
    for start in range(0, n_samples, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, n_samples)
        chunk = eeg_data[start:end].astype(np.float32)
        outlet.push_chunk(chunk.tolist())

    time.sleep(0.5)

    streams = resolve_byprop('name', 'SimulatedEEG', timeout=2)
    inlet = StreamInlet(streams[0])

    received = []
    total_received = 0
    while total_received < n_samples:
        chunk_data, _ = inlet.pull_chunk(timeout=1.0)
        if not chunk_data:
            break
        received.extend(chunk_data)
        total_received += len(chunk_data)

    received_data = np.array(received[:n_samples])
    used_real_lsl = True
    print('Real LSL stream used successfully.')
except Exception as e:
    print(f'LSL not available ({e}), falling back to simulation.')
    received_data = eeg_data.copy()
    used_real_lsl = False
    print('Simulation mode used (data copied directly).')

print(f'Received data shape: {received_data.shape}')

## 5. Interactive plot

**What to look for:**
- The top plot shows the original 4-channel signal, offset vertically for visibility
- The bottom plot shows the received signal via LSL (or simulation), also offset
- The two plots should look **identical** — LSL preserves the data exactly
- Each channel is shifted by a constant offset so they don't overlap

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(N_PLOT, eeg_data.shape[0])
x = np.arange(n_plot)
offsets = [0, 100, 200, 300]
mode_label = 'Real LSL' if used_real_lsl else 'Simulation'

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original 4-Channel Signal',
                                    f'Received via LSL ({mode_label})'))

for i in range(CHANNEL_COUNT):
    fig.add_trace(go.Scatter(x=x, y=eeg_data[:n_plot, i] + offsets[i],
                             name=f'{ch_names[i]} (orig)',
                             line=dict(width=0.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=received_data[:n_plot, i] + offsets[i],
                             name=f'{ch_names[i]} (recv)',
                             line=dict(width=0.5)), row=2, col=1)

fig.update_layout(height=700, title_text='LSL Stream - Outlet and Inlet Demo',
                  xaxis2_title='Sample index',
                  yaxis_title='Amplitude + offset (uV)',
                  yaxis2_title='Amplitude + offset (uV)')
fig.show()

## What did we learn?

- **LSL (Lab Streaming Layer)** is a protocol for streaming real-time EEG data between applications
- An **outlet** publishes data, and an **inlet** receives it — they can be in different processes or machines
- Data is pushed in **chunks** for efficiency, not one sample at a time
- When pylsl is unavailable, a **simulation fallback** demonstrates the same concept
- LSL is widely used in BCI research for connecting EEG hardware to analysis software